<a href="https://colab.research.google.com/github/hariszidan-ui/ProjectARKA/blob/main/ProjectARKA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Dari Sistem Persamaan Linear ke Dimensi Tersembunyi

Catatan dari pembuat:

Notebook ini terdiri atas dua bagian. Bagian A merupakan soal pemanasan berbentuk **fill in the blanks** untuk memperkenalkan hubungan antara regresi linear dan Sistem Persamaan Linear (SPL) yang telah dipelajari di kelas. Bagian B merupakan task supervised learning. Silakan dieksplorasi!

## Tujuan Pembelajaran

Anda akan menggunakan rank, null space, least squares, reduksi dimensi, perubahan basis, dan feature mapping dalam satu alur supervised learning. Pastikan kode dapat dijalankan ulang dan setiap keputusan didukung oleh hasil eksperimen.

## Setup dan Import

In [52]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

np.set_printoptions(suppress=True, precision=6)
RANDOM_STATE = 42


In [76]:
import os
REPO_ROOT = Path("/content/ProjectArka")
if not REPO_ROOT.exists():
    print("Fetching repository.")
    os.system("git clone -b haris --single-branch https://github.com/hariszidan-ui/ProjectARKA.git")

%cd /content/ProjectARKA
%load_ext autoreload
%autoreload 2


os.system("git pull")
from src import haris as h

/content/ProjectARKA


ModuleNotFoundError: No module named 'imp'

## Fungsi Evaluasi

Gunakan fungsi berikut untuk menghitung RMSE pada data validasi dengan definisi yang sama seperti leaderboard Kaggle. Nilai yang lebih rendah menunjukkan prediksi yang lebih baik, sedangkan prediksi sempurna memperoleh nilai 0. Target untuk `test.csv` tidak tersedia.

In [54]:
def rmse(y_true, y_pred):
    """Menghitung Root Mean Squared Error (RMSE)."""
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    if y_true.shape != y_pred.shape:
        raise ValueError("y_true dan y_pred harus memiliki bentuk yang sama.")
    if y_true.size == 0:
        raise ValueError("Data harus memiliki sekurang-kurangnya satu observasi.")
    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError("Target dan prediksi hanya boleh memuat nilai hingga.")

    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


Anda juga dapat menggunakan fungsi `root_mean_squared_error` bawaan scikit-learn:

```python
error = root_mean_squared_error(y_true, y_pred)
```

# Bagian A: Tutorial SPL dengan Linear Regression

Bagian ini menggunakan tiga dataset untuk mempelajari SPL dengan solusi unik, tak hingga banyak solusi, dan sistem inkonsisten. Setiap baris merepresentasikan satu persamaan, setiap kolom fitur merepresentasikan satu variabel, dan kolom target menjadi ruas kanan $y$.

Gunakan `LinearRegression(fit_intercept=False)` karena SPL sudah berbentuk $X\beta=y$. Menambahkan intercept berarti menambahkan variabel baru ke sistem.

## A0. Memuat Dataset SPL

In [75]:
"""
Handle competition download by manually inputting KAGGLE_API_TOKEN
"""

from getpass import getpass
from dotenv import load_dotenv

DOTENV_PATH = REPO_ROOT / '.env'
h.load_KAGGLE_API_TOKEN(DOTENV_PATH)

!kaggle -p '/kaggle/input' -c 'project-a-r-k-a-hidden-dimensions'

AttributeError: module 'src.haris' has no attribute 'load_KAGGLE_API_TOKEN'

In [56]:
!dir /kaggle/input

In [57]:
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
REQUIRED_FILES = {
    'train.csv',
    'test.csv',
    'calibration.csv',
    'sample_submission.csv',
    'spl_01_harga_rumah.csv',
    'spl_02_pendapatan_supermarket.csv',
    'spl_03_risiko_fraud.csv',
}

DATA_DIR = next(
    (
        path for path in KAGGLE_INPUT_ROOT.iterdir()
        if path.is_dir()
        and REQUIRED_FILES.issubset({file.name for file in path.iterdir()})
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        'Dataset kompetisi tidak ditemukan di /kaggle/input. '
        'Tambahkan dataset kompetisi ke notebook, lalu jalankan kembali sel ini.'
    )


FileNotFoundError: Dataset kompetisi tidak ditemukan di /kaggle/input. Tambahkan dataset kompetisi ke notebook, lalu jalankan kembali sel ini.

In [ ]:
def dataframe_to_system(dataframe, target_column):
    feature_columns = [
        column for column in dataframe.columns
        if column != target_column
    ]
    X = dataframe[feature_columns].to_numpy(dtype=float)
    y = dataframe[target_column].to_numpy(dtype=float)
    return X, y, feature_columns


unique_df = pd.read_csv(DATA_DIR / 'spl_01_harga_rumah.csv')
infinite_df = pd.read_csv(DATA_DIR / 'spl_02_pendapatan_supermarket.csv')
inconsistent_df = pd.read_csv(DATA_DIR / 'spl_03_risiko_fraud.csv')

X_unique, y_unique, unique_features = dataframe_to_system(
    unique_df,
    'price_usd_thousand',
)
X_infinite, y_infinite, infinite_features = dataframe_to_system(
    infinite_df,
    'daily_revenue_usd',
)
X_inconsistent, y_inconsistent, inconsistent_features = dataframe_to_system(
    inconsistent_df,
    'fraud_risk_score',
)

for name, X, feature_names in [
    ('Solusi unik', X_unique, unique_features),
    ('Tak hingga banyak solusi', X_infinite, infinite_features),
    ('Inkonsisten', X_inconsistent, inconsistent_features),
]:
    print(f'{name}: {X.shape[0]} persamaan, {X.shape[1]} variabel')

display(unique_df.head(3))
display(infinite_df.head(3))
display(inconsistent_df.head(3))


## A1. Solusi Unik

Dataset harga rumah memiliki satu persamaan untuk setiap observasi dan satu variabel untuk setiap fitur. Lengkapi kode berikut, lalu bandingkan rank $X$ dengan rank augmented matrix $[X\mid y]$.

In [ ]:
model_unique = LinearRegression(fit_intercept=____)
model_unique.fit(____, ____)

beta_unique = model_unique.____
prediction_unique = model_unique.____(X_unique)

rank_X_unique = np.linalg.matrix_rank(____)
augmented_unique = np.column_stack([____, ____])
rank_augmented_unique = np.linalg.matrix_rank(____)
residual_unique = np.linalg.norm(____ - ____)

print('beta:', beta_unique)
print('rank X:', rank_X_unique)
print('rank augmented:', rank_augmented_unique)
print('residual norm:', residual_unique)


**Pertanyaan:** Mengapa residual nol dan rank yang sama dengan jumlah variabel menunjukkan solusi unik?

## A2. Tak Hingga Banyak Solusi

Dataset pendapatan supermarket memiliki lebih banyak variabel daripada persamaan independen. `LinearRegression` menghasilkan salah satu solusi. Temukan vektor null space untuk membentuk solusi lain.

In [ ]:
model_infinite = LinearRegression(fit_intercept=____)
model_infinite.fit(____, ____)
beta_infinite = model_infinite.____

_, singular_values, Vt = np.linalg.svd(____, full_matrices=True)
null_vector = Vt[____]
beta_alternative = beta_infinite + 5 * ____

rank_X_infinite = np.linalg.matrix_rank(____)
rank_augmented_infinite = np.linalg.matrix_rank(
    np.column_stack([____, ____])
)

print('beta from model:', beta_infinite)
print('null-space check:', np.linalg.norm(X_infinite @ null_vector))
print('first residual:', np.linalg.norm(X_infinite @ beta_infinite - y_infinite))
print('alternative residual:', np.linalg.norm(X_infinite @ beta_alternative - y_infinite))
print('ranks:', rank_X_infinite, rank_augmented_infinite)


**Pertanyaan:** Mengapa dua koefisien berbeda dapat menghasilkan prediksi yang sama?

## A3. Sistem Inkonsisten

Pada sistem inkonsisten, regresi menghasilkan solusi least squares, bukan solusi eksak SPL. Lengkapi kode berikut untuk membandingkan rank matriks fitur, rank augmented matrix, dan residual model.

In [ ]:
model_inconsistent = LinearRegression(fit_intercept=____)
model_inconsistent.fit(____, ____)

beta_inconsistent = model_inconsistent.____
prediction_inconsistent = model_inconsistent.____(X_inconsistent)
residual_vector = ____ - ____

rank_X_inconsistent = np.linalg.matrix_rank(____)
rank_augmented_inconsistent = np.linalg.matrix_rank(
    np.column_stack([____, ____])
)
normal_equation_check = X_inconsistent.T @ ____

print('beta least squares:', beta_inconsistent)
print('residual norm:', np.linalg.norm(residual_vector))
print('ranks:', rank_X_inconsistent, rank_augmented_inconsistent)
print('X^T residual:', normal_equation_check)


**Pertanyaan:** Mengapa koefisien tetap tersedia ketika $X\beta=y$ tidak mempunyai solusi eksak?

# Bagian B: Project A.R.K.A.,

A.R.K.A. adalah rover pencarian dan penyelamatan. **ATLAS**, **BOREALIS**, dan **CYGNUS** merupakan tiga profil firmware sensor. `train.csv` memuat misi berlabel yang direkam menggunakan ATLAS, sedangkan `test.csv` memuat misi tanpa label yang direkam menggunakan BOREALIS dan CYGNUS. Setiap profil merepresentasikan keadaan fisik rover melalui koordinat sensor yang berbeda.

`calibration.csv` memuat pengukuran misi kalibrasi yang sama di bawah ketiga profil. Baris dengan `calibration_id` yang sama merepresentasikan keadaan fisik rover yang sama. File ini tidak memiliki kolom `energy_consumption`, tetapi dapat digunakan untuk mempelajari hubungan antarrepresentasi sensor.

Prediksi dinilai menggunakan Root Mean Squared Error (RMSE). Nilai yang lebih rendah menunjukkan prediksi yang lebih baik, sedangkan prediksi sempurna memperoleh nilai 0.

## B1. Memuat Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')

test_df = pd.read_csv(DATA_DIR / 'test.csv')
calibration_df = pd.read_csv(DATA_DIR / 'calibration.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'energy_consumption'
ID_COLUMN = 'id'
FIRMWARE_COLUMN = 'firmware'
SENSOR_COLUMNS = [column for column in train_df if column.startswith('sensor_')]

print('Train:', train_df.shape)
print('Test:', test_df.shape)
print('Calibration:', calibration_df.shape)
display(train_df.head())
display(calibration_df.head())


## B2. Exploratory Data Analysis

Periksa kualitas data, distribusi target, perbedaan profil firmware, pasangan `calibration_id`, serta struktur matriks sensor. Gunakan ringkasan numerik dan visualisasi yang sesuai. Jelaskan keterkaitan setiap temuan dengan model yang akan dibangun.

In [ ]:
# TODO: eksplorasi data.
display(train_df.describe())
print(train_df[FIRMWARE_COLUMN].value_counts())
print(test_df[FIRMWARE_COLUMN].value_counts())
print(calibration_df.groupby('calibration_id')[FIRMWARE_COLUMN].nunique().describe())


### Analisis EDA

Tuliskan temuan dan hipotesis mengenai hubungan antara data training, data test, dan data kalibrasi.

## B3. Baseline Supervised Learning

Baseline berikut menggunakan regresi linear pada validation split dari data ATLAS. Gunakan hasilnya sebagai titik awal untuk mengevaluasi pendekatan berikutnya. Skor validasi pada ATLAS belum menunjukkan kinerja model pada BOREALIS dan CYGNUS.

In [ ]:
X = train_df[SENSOR_COLUMNS]
y = train_df[TARGET]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

baseline = LinearRegression()
baseline.fit(X_train, y_train)
baseline_prediction = baseline.predict(X_valid)

print('RMSE:', mean_squared_error(y_valid, baseline_prediction) ** 0.5)
print('R²:', r2_score(y_valid, baseline_prediction))
print('Competition RMSE:', rmse(y_valid, baseline_prediction))


## B3.1 Submission Valid Pertama

Prediktor rata-rata target menghasilkan submission valid untuk memeriksa format dan alur penyimpanan CSV. Gunakan skor leaderboard-nya sebagai baseline awal.

In [ ]:
mean_baseline_value = train_df[TARGET].mean()

baseline_submission = sample_submission.copy()
baseline_submission[TARGET] = mean_baseline_value

assert baseline_submission[ID_COLUMN].equals(test_df[ID_COLUMN])
assert baseline_submission[TARGET].notna().all()

baseline_submission.to_csv(
    'baseline_mean_submission.csv',
    index=False,
)

print('Mean prediction:', mean_baseline_value)
display(baseline_submission.head())


## B4. Feature Engineering dan Eksperimen

Rancang representasi yang dapat digunakan untuk ATLAS, BOREALIS, dan CYGNUS. Gunakan pasangan misi pada `calibration.csv` untuk mengevaluasi hubungan antarprofil. Selanjutnya, lakukan eksperimen dengan pemetaan fitur dan model supervised learning.

Catat representasi, jumlah fitur, RMSE validasi, $R^2$ validasi, dan alasan matematis untuk setiap eksperimen.

In [ ]:
experiment_results = pd.DataFrame(columns=[
    'experiment', 'representation', 'n_features', 'rmse', 'r2'
])

# TODO: implementasikan feature engineering dan eksperimen terbaik.
experiment_results


### Analisis Eksperimen

Jelaskan bagaimana struktur matriks, hubungan antarprofil, dan feature mapping memengaruhi hasil. Bandingkan metode berdasarkan hasil validasi dan alasan matematisnya.

## B5. Model Final dan Submission

In [ ]:
# TODO: latih seluruh pipeline pada train_df dan prediksi test_df.
test_prediction = None

# Silakan uncomment setelah test_prediction tersedia.
# submission = sample_submission.copy()
# submission[TARGET] = test_prediction
# assert submission[ID_COLUMN].equals(test_df[ID_COLUMN])
# assert submission[TARGET].notna().all()
# submission.to_csv('submission.csv', index=False)
# display(submission.head())
